<a href="https://colab.research.google.com/github/amlucky0/AI-Driven-Citizen-Grievance-System/blob/main/Project1_Week3_Sentiment_Urgency_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Dependencies install karein
!pip install transformers torch scikit-learn

In [2]:
# Cell 2: Sentiment aur Priority logic test karein
from transformers import pipeline

# RoBERTa Model load karein (Colab ke GPU par fast chalega)
classifier = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

def calculate_priority(text, dept_confidence=0.90):
    res = classifier(text)[0]
    label, score = res['label'], res['score']

    # Business Logic
    if label == 'negative' and score > 0.75:
        tag, severity = 'Critical/Urgent', 3.0
    elif label == 'negative':
        tag, severity = 'Negative', 2.0
    else:
        tag, severity = 'Neutral', 1.0

    priority_score = (severity * score * 0.7) + (dept_confidence * 0.3)
    return tag, round(priority_score, 2)

# Test karein
print(calculate_priority("The street lights are not working, it's dangerous!"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

('Critical/Urgent', 2.18)


In [3]:
# Cell 3: FastAPI aur Web Tunneling tools install karein
!pip install fastapi uvicorn pydantic nest-asyncio

In [4]:
# Cell 4: app.py file write karein directly Colab se
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class GrievanceInput(BaseModel):
    text: str

@app.post("/triage/")
def triage(data: GrievanceInput):
    # Short mapping for quick testing
    return {"predicted_department": "Sanitation", "sentiment": "Negative", "priority_score": 2.1}

Writing app.py


In [5]:
# Cell 5: Background me server start karein
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()
def run_app():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Server ko thread me chalayein taaki Colab block na ho
Thread(target=run_app).start()

Exception in thread Thread-5 (run_app):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_883/1112601183.py", line 8, in run_app
NameError: name 'app' is not defined


In [6]:
# CODE FOR WEEK 3: INTEGRATED PRIORITY AND TESTING ENGINE

def process_grievance_urgency(text, predicted_dept, dept_confidence):
    """
    Dono models (Week 2 aur Week 3) ke output ko integrate karne ka core coordination function.
    """
    # Sentiment Analysis execute karein
    sentiment_result = classifier(text)[0]
    label = sentiment_result['label']
    confidence = sentiment_result['score']

    # 1. Text Tone aur Severity Weight assign karein (Enterprise Logic)
    if label == 'negative' and confidence > 0.75:
        sentiment_tag = "Critical/Urgent"
        severity_weight = 3.0
    elif label == 'negative':
        sentiment_tag = "Negative"
        severity_weight = 2.0
    elif label == 'neutral':
        sentiment_tag = "Neutral"
        severity_weight = 1.0
    else:
        sentiment_tag = "Positive"
        severity_weight = 0.5

    # 2. Priority Score Formula (70% Sentiment Weight + 30% Department Confidence)
    # Yeh formula dynamic data tracking ko handle karta hai
    calculated_priority = (severity_weight * confidence * 0.7) + (dept_confidence * 0.3)

    return {
        "text": text,
        "department": predicted_dept,
        "sentiment_tone": sentiment_tag,
        "priority_score": round(calculated_priority, 2)
    }

# Synthetic Test Dataset par check karte hain (Jaise 311 service request data)
test_complaints = [
    {"text": "Shortage of water from last 4 days in block C. Please help!", "dept": "Water Supply", "conf": 0.92},
    {"text": "Street lights are blinking but working fine overall.", "dept": "Electricity", "conf": 0.85},
    {"text": "Thank you for fixing the potholes on main road so quickly.", "dept": "Roads & Transport", "conf": 0.95}
]

# Run Pipeline
print("--- WEEK 3 EVALUATION RESULTS ---")
for complaint in test_complaints:
    result = process_grievance_urgency(complaint['text'], complaint['dept'], complaint['conf'])
    print(f"\nDepartment: {result['department']} | Tone: {result['sentiment_tone']} | Priority Score: {result['priority_score']}")
    print(f"Complaint: {result['text']}")

--- WEEK 3 EVALUATION RESULTS ---

Department: Water Supply | Tone: Critical/Urgent | Priority Score: 2.0
Complaint: Shortage of water from last 4 days in block C. Please help!

Department: Electricity | Tone: Positive | Priority Score: 0.45
Complaint: Street lights are blinking but working fine overall.

Department: Roads & Transport | Tone: Positive | Priority Score: 0.54
Complaint: Thank you for fixing the potholes on main road so quickly.


In [7]:
# Cell: Week 4 Dependencies
!pip install fastapi uvicorn pydantic nest-asyncio requests

In [9]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import uvicorn

# 1. FastAPI Initialize karein
app = FastAPI(title="AI-Driven Citizen Grievance Redressal System")

# 2. Week 3 ka Transformer pipeline backend par load karein
try:
    sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
except Exception as e:
    sentiment_pipeline = None

# 3. Request (Input) JSON structure design karein
class GrievanceInput(BaseModel):
    text: str

# 4. Response (Output) JSON structure design karein
class GrievanceOutput(BaseModel):
    predicted_department: str
    sentiment: str
    priority_score: float

@app.get("/")
def home():
    return {"status": "Active", "system": "Infotact Grievance API Portal"}

# 5. Core Triage Endpoint jo JSON payload accept karega
@app.post("/triage/", response_model=GrievanceOutput)
def triage_complaint(data: GrievanceInput):
    if not data.text.strip():
        raise HTTPException(status_code=400, detail="Complaint text cannot be empty.")

    # Week 2 Multi-class classification simulation (e.g., Water, Electricity, Roads)
    # Production mein yahan joblib.load() se model load hota hai
    mock_department = "Water Supply"
    mock_dept_confidence = 0.88

    # Week 3 Sentiment Processing
    if sentiment_pipeline:
        res = sentiment_pipeline(data.text)[0]
        label = res['label']
        score = res['score']
    else:
        label, score = 'neutral', 0.5

    # Urgency Severity weights assignment
    if label == 'negative' and score > 0.70:
        sentiment_tag = "Critical/Urgent"
        severity = 3.0
    elif label == 'negative':
        sentiment_tag = "Negative"
        severity = 2.0
    else:
        sentiment_tag = "Neutral"
        severity = 1.0

    # Integrated Final Priority Score Formula (70% Sentiment + 30% Classification Confidence)
    final_priority = (severity * score * 0.7) + (mock_dept_confidence * 0.3)

    return {
        "predicted_department": mock_department,
        "sentiment": sentiment_tag,
        "priority_score": round(final_priority, 2)
    }

Overwriting app.py


In [11]:
# Cell: Background Server Activation (UPDATED FIX)
import nest_asyncio
import uvicorn
from threading import Thread

# Fix: app.py file se 'app' object ko Colab ki current memory mein import karna zaroori hai
try:
    from app import app
    print("Success: 'app' object successfully loaded from app.py!")
except ImportError:
    print("Error: Make sure you successfully ran the Step 2 cell to create app.py first.")

nest_asyncio.apply()

def run_server():
    # Server ko localhost port 8000 par start karein
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Threading se server background mein bina cell block kiye chalega
server_thread = Thread(target=run_server)
server_thread.start()
print("FastAPI local server successfully triggered on http://127.0.0.1:8000")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success: 'app' object successfully loaded from app.py!
FastAPI local server successfully triggered on http://127.0.0.1:8000


In [12]:
# Cell: Background Server Activation
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run_server():
    # Server ko localhost port 8000 par start karein
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Threading se server background mein bina cell block kiye chalega
server_thread = Thread(target=run_server)
server_thread.start()
print("FastAPI local server successfully triggered on http://127.0.0.1:8000")

FastAPI local server successfully triggered on http://127.0.0.1:8000


INFO:     Started server process [883]
INFO:     Waiting for application startup.


In [13]:
# Cell: API Request Test (Step 4)
import requests
import json

# Python requests ke zariye local endpoint par payload bhejte hain
url = "http://127.0.0.1:8000/triage/"
payload = {"text": "Heavy sewage blockage in our colony area from past two weeks. Smells terrible!"}
headers = {"Content-Type": "application/json"}

try:
    response = requests.post(url, data=json.dumps(payload), headers=headers)
    print("--- API LIVE RESPONSE TEST ---")
    print(json.dumps(response.json(), indent=4))
except Exception as e:
    print("Error connecting to server:", e)

INFO:     127.0.0.1:46424 - "POST /triage/ HTTP/1.1" 200 OK
--- API LIVE RESPONSE TEST ---
{
    "predicted_department": "Water Supply",
    "sentiment": "Critical/Urgent",
    "priority_score": 2.23
}
